## The State of Tax Justice: Estimate misalignment
- Author: Alison Schultz, based on Javier Garcia Bernado's work
- Created: 4 August 2023
- Last updated: 22 September 2024

**Description**
- This notebook is the third out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis used the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
    - Details on the misalignment method and its background can be found here: https://www.sciencedirect.com/science/article/pii/S0305750X23003455. 
    - The working paper version is here: https://www.econstor.eu/bitstream/10419/286362/1/wp-2023-33.pdf 

- This notebook estimates profit misalignment based on different formulas. It uses the dataset **"data/final/cbcr_main.csv"** (for the estimation with imputed values) or the dataset **"data/final/cbcr_main_noimputation_allsubgroupsonly.csv"** (for the estimation without imputed values). 

**Outline**
1. Define misalignment
2. Calculate misalignment for sample with full information.
3. Calculate misalignment for samples with imputed data and aggregate results. 

**To dos before running this notebook**
- Run the notebooks 1_clean and 2_impute_missings. Note the requirements and instructions given in these notebooks.

**To dos in this notebook**

Change the formula to any required formula in each section and adapt the output name of the csvs. The formulas I have used are like follows, where sales refer to unrelated party revenues and assets to tangible assets excluding cash.

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]

- Adjust the input path in 5.1 to the bootstrapped sample you use

## 0. Load packages

In [7]:
import pandas as pd
import numpy as np
import tjn_tools
from config import *

#Show all columns
#pd.set_option('display.max_columns', None)
# Select a particular iso_parent
#cbcr_sample = cbcr_sample[cbcr_sample['iso_parent'] == 'AUT']
#cbcr_sample

## 1. Define misalignment

In [15]:
def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",
                                         'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[.5, 0, 0, .5, 0, 0, 0, 0],
                           profit_var='profit_loss_before_income_tax_corrected',
                           etr_max=0.15): 

    # Create variable with positive profits only for calculating shares
    cbcr_data['profit_var_pos'] = cbcr_data[profit_var]
    cbcr_data.loc[cbcr_data[profit_var] < 0, 'profit_var_pos'] = 0
    cbcr_data['share_profit'] = cbcr_data['profit_var_pos'] / cbcr_data.groupby('iso_parent')['profit_var_pos'].transform('sum')

    # Calculate weighted shares of economic activity
    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f"share_{var}")
            actual_weights.append(weights[i])
            cbcr_data.loc[cbcr_data[var] < 0, var] = 0  # Set economic activity measure to zero if negative
            cbcr_data[f"share_{var}"] = cbcr_data[var] / cbcr_data.groupby('iso_parent')[var].transform('sum')

    # Calculate the share of economic activity
    cbcr_data["share_economy_partner_of_parent"] = (cbcr_data.loc[:, actual_variables] * actual_weights).sum(1, min_count=len(actual_weights))
    # Set economic activity to 1% for those jurisdictions without economic activity but with reported profits
    cbcr_data.loc[(cbcr_data["share_economy_partner_of_parent"] == 0) & (cbcr_data[profit_var] > 0), "share_economy_partner_of_parent"] = 0.01

    # Normalize the economic activity shares to sum to 1
    cbcr_data["share_economy_partner_of_parent"] = cbcr_data["share_economy_partner_of_parent"] / cbcr_data.groupby('iso_parent')["share_economy_partner_of_parent"].transform('sum')

    # Calculate theoretical profit and misaligned profit
    cbcr_data["theoretical_profit"] = cbcr_data["share_economy_partner_of_parent"] * cbcr_data.groupby('iso_parent')[profit_var].transform('sum')
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var] - cbcr_data["theoretical_profit"]

    # Set positive misaligned profits to 0 if ETR exceeds the threshold (etr_max)
    cbcr_data.loc[((cbcr_data["misaligned_profit"] > 0) & (cbcr_data["etr_average_corrected"] > etr_max)), "misaligned_profit"] = 0

    # Adjust misalignment per 'iso_parent'
    def adjust_misalignment(group):
        total_negative_misalignment = group.loc[group["misaligned_profit"] < 0, "misaligned_profit"].sum()
        total_positive_misalignment = group.loc[group["misaligned_profit"] > 0, "misaligned_profit"].sum()
        
        # Adjust negative misalignments to balance positive misalignments within each 'iso_parent'
        if total_negative_misalignment != 0:
            factor = - total_positive_misalignment / total_negative_misalignment
            group.loc[group["misaligned_profit"] < 0, "misaligned_profit"] *= factor
        
        return group

    # Apply the adjustment by grouping by 'iso_parent'
    cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(adjust_misalignment, include_groups=False).reset_index(drop=True)

    return cbcr_data

In [16]:
def calculate_misalignment_simplified(cbcr_data,
                                       formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",
                                                     'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip'],
                                       weights=[.5, 0, 0, .5, 0, 0, 0, 0],
                                       profit_var='profit_loss_before_income_tax_corrected',
                                       etr_max=0.15):
    
    # Create variable with positive profits only for calculating shares
    cbcr_data['profit_var_pos'] = cbcr_data[profit_var].clip(lower=0)
    cbcr_data['share_profit'] = cbcr_data['profit_var_pos'] / cbcr_data['profit_var_pos'].sum()

    # Calculate weighted shares of economic activity
    actual_weights = []
    actual_variables = []
    
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f"share_{var}")
            actual_weights.append(weights[i])
            cbcr_data.loc[cbcr_data[var] < 0, var] = 0  # Set economic activity measure to zero if negative
            cbcr_data[f"share_{var}"] = cbcr_data[var] / cbcr_data[var].sum()

    # Calculate the share of economic activity globally
    cbcr_data["share_economy_global"] = (cbcr_data.loc[:, actual_variables] * actual_weights).sum(axis=1)
    
    # Set economic activity to 1% for those jurisdictions without economic activity but with reported profits
    cbcr_data.loc[(cbcr_data["share_economy_global"] == 0) & (cbcr_data[profit_var] > 0), "share_economy_global"] = 0.01

    # Normalize the economic activity shares to sum to 1 globally
    cbcr_data["share_economy_global"] = cbcr_data["share_economy_global"] / cbcr_data["share_economy_global"].sum()

    # Calculate theoretical profit and misaligned profit globally
    cbcr_data["theoretical_profit"] = cbcr_data["share_economy_global"] * cbcr_data[profit_var].sum()
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var] - cbcr_data["theoretical_profit"]

    # Set positive misaligned profits to 0 if ETR exceeds the threshold (etr_max)
    cbcr_data.loc[((cbcr_data["misaligned_profit"] > 0) & (cbcr_data["etr_average_corrected"] > etr_max)), "misaligned_profit"] = 0

    # Adjust negative misalignment to fit positive misalignment
    total_negative_misalignment = cbcr_data.loc[cbcr_data["misaligned_profit"] < 0, "misaligned_profit"].sum()
    total_positive_misalignment = cbcr_data.loc[cbcr_data["misaligned_profit"] > 0, "misaligned_profit"].sum()

    # Adjust negative misalignments to balance positive misalignments within each 'iso_parent'
    factor = - total_positive_misalignment / total_negative_misalignment
    cbcr_data.loc[cbcr_data["misaligned_profit"] < 0, "misaligned_profit"] *= factor
        
    return cbcr_data


## 2. Calculate misalignment for sample with full information

### 2.1 Import data
- Import data without imputed values. This is only the data from the sample of reporting countries that actually is reported on a country basis, i.e. excluding aggregated country groups and data from reporting countries that do not report on a country by country basis, but just by continents.

In [17]:
cbcr_sample = pd.read_csv(f'{data_final}/cbcr_dataset_total_allsubgroups.csv')

# Exclude country groups
cbcr_sample = cbcr_sample[~cbcr_sample['iso_partner'].isin(non_countries)]



### 2.2 Exclude countries that do not report truly country-by-country


- In the 2024 data, the following reporting countries do not report country-by-country. We exclude those from the "clean" analysis where we only use values that are actually in the data.
    - Austria: Only continents in all years
    - Czechia: Only Czechia versus rest of the world from 2019 to 2021
    - Finland: Only Finland and rest of the world between 2016 and 2018 and Finland and continents between 2019 and 2021
    - Greece: Only Greece and continents between 2017 and 2019
    - Hungary: Only Hungary versus rest of the world between 2018 and 2021
    - Isle of Man: Only continents between 2017 and 2020
    - Ireland: Only Ireland versus rest of the world in all years
    - Korea: Only Korea and rest of the world betweem 2016 and 2018 and Korea and continents between 2019 and 2021
    - Macau: Only Macau versus rest of the world between 2019 and 2021
    - Mauritius: Only Mauritius and continents between 2019 and 2021
    - Morocco: Only Morocco and continents in 2021
    - Netherlands: Only Netherlands versus rest of the world between 2016 and 2017
    - Norway: Only Norway and continents 2016 and 2017
    - New Zealand: Only New Zealand versus rest of the world between 2018 and 2021
    - Poland: Only Poland and continents 2019 to 2021
    - Sweden: Only Sweden and continents in all years
    - United Kingdom: Only UK and continents between 2017 and 2021

In [18]:
# Define the conditions for exclusion
exclusion_conditions = [
    ('AUT', 2016, 2021),                # Austria: all years
    ('CZE', 2019, 2021),                # Czechia: from 2019 to 2021
    ('FIN', 2016, 2021),                # Finland: all years
    ('GRC', 2017, 2019),                # Greece: between 2017 and 2019
    ('HUN', 2018, 2021),                # Hungary: between 2018 and 2021
    ('IMN', 2017, 2020),                # Isle of Man: between 2017 and 2020
    ('IRL', 2016, 2021),                # Ireland: all years
    ('KOR', 2016, 2021),                # Korea: all years
    ('MAC', 2019, 2021),                # Macau: between 2019 and 2021
    ('MUS', 2019, 2021),                # Mauritius: between 2019 and 2021
    ('MAR', 2021, 2021),                # Morocco: 2021
    ('NLD', 2016, 2017),                # Netherlands: between 2016 and 2017
    ('NOR', 2016, 2017),                # Norway: 2016 and 2017
    ('NZL', 2018, 2021),                # New Zealand: between 2018 and 2021
    ('POL', 2019, 2021),                # Poland: 2019 to 2021
    ('SWE', 2016, 2021),                # Sweden: all years
    ('GBR', 2017, 2021)                 # United Kingdom: between 2017 and 2021
]

# Iterate through the exclusion conditions
for iso_parent, start_year, end_year in exclusion_conditions:
    cbcr_sample = cbcr_sample[~((cbcr_sample['iso_parent'] == iso_parent) & 
                                     (cbcr_sample['year'].between(start_year, end_year)))]
    

### 2.3 Calculate misalignment for sample countries with full information

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]


In [19]:
# Initialize a list to store the aggregate results
results_sample = []

for year in range(first_year, first_year + n_years):
    #print(f"For correct reporters: Total profit shifted in USD mn {year}")
    
    misalignment_year = cbcr_sample[cbcr_sample['year'] == year].copy()
    misalignment_year = calculate_misalignment(misalignment_year, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

    # Keep only the first occurrence of these unique variables for each 'iso_partner'
    unique_columns = misalignment_year.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

    # Perform the groupby operation on 'iso_partner'
    country_results_year = misalignment_year.groupby(['iso_partner']).agg(
        negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
        theoretical_profit=('theoretical_profit', 'sum'),
        reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
    ).reset_index()

    # Convert results to millions
    country_results_year['negative_misalignment'] = -country_results_year['negative_misalignment'] / 1e6
    country_results_year['positive_misalignment'] = country_results_year['positive_misalignment'] / 1e6
    country_results_year['theoretical_profit'] = country_results_year['theoretical_profit'] / 1e6
    country_results_year['reported_profit'] = country_results_year['reported_profit'] / 1e6

    # Merge the unique columns back into the result
    country_results_year = country_results_year.merge(unique_columns, on='iso_partner', how='left')

    # Calculate other relevant variables
    country_results_year['tax_revenue_loss'] = country_results_year['negative_misalignment'] * country_results_year['cit']
    country_results_year['tax_revenue_gain'] = country_results_year['positive_misalignment'] * country_results_year['etr_average_corrected']

    country_results_year['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
        country_results_year['gvt_health_expenditure'] == 0, 
        np.nan, 
        country_results_year['tax_revenue_loss'] / (country_results_year['gvt_health_expenditure'] / 1e6)
    )
    
    country_results_year['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
        country_results_year['tax_revenue_current_usd'] == 0, 
        np.nan, 
        country_results_year['tax_revenue_loss'] / (country_results_year['tax_revenue_current_usd'] / 1e6)
    )

    # Calculate totals
    total_positive_misalignment = country_results_year['positive_misalignment'].sum()
    total_negative_misalignment = country_results_year['negative_misalignment'].sum()
    total_profits = country_results_year['reported_profit'].sum()
    misaligned_of_total_profits = total_positive_misalignment / total_profits
    total_tax_revenue_loss = country_results_year['tax_revenue_loss'].sum()
    total_tax_revenue_gain = country_results_year['tax_revenue_gain'].sum()
    average_tax_revenue_loss_pct_of_gvt_health_expenditure = country_results_year['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
    average_tax_revenue_loss_pct_of_total_tax_revenues = country_results_year['tax_revenue_loss_pct_of_total_tax_revenues'].mean()


    print(f"Year {year}: Positive Misalignment: {total_positive_misalignment}, Negative Misalignment: {total_negative_misalignment}, Shifted of total profits: {misaligned_of_total_profits}, "
          f"Total tax revenue loss: {total_tax_revenue_loss}, Total tax revenue gain: {total_tax_revenue_gain}")

    # Calculate countries' fractions of totals
    country_results_year['tax_revenue_loss_caused_pct_of_total'] = country_results_year['positive_misalignment'] / total_positive_misalignment
    country_results_year['tax_revenue_loss_caused_usd'] = country_results_year['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss
    country_results_year['tax_revenue_loss_suffered_pct_of_total'] = country_results_year['tax_revenue_loss'] / total_tax_revenue_loss

    #country_results_year = country_results_year[['iso_partner', 'partner_jurisdiction', 'negative_misalignment',
    #   'tax_revenue_loss', 'tax_revenue_loss_suffered_pct_of_total', 'tax_revenue_loss_pct_of_gvt_health_expenditure',
    #   'tax_revenue_loss_pct_of_total_tax_revenues', 'positive_misalignment', 'tax_revenue_gain', 
    #   'tax_revenue_loss_caused_usd', 'tax_revenue_loss_caused_pct_of_total', 'etr_average_corrected', 'cit',
    #   'region_tjn', 'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

    country_results_year = country_results_year.sort_values(by='iso_partner')
    country_results_year.to_csv(f'{output_tables}/Sample_with_original_data_only/SOTJ_sample_{year}.csv', index=False) # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE
    
    # Append aggregate results to the list
    results_sample.append({
        'year': year,
        'total_positive_misalignment': total_positive_misalignment,
        'total_negative_misalignment': total_negative_misalignment,
        'total_profits': total_profits,
        'misaligned_of_total_profits': misaligned_of_total_profits,
        'total_tax_revenue_loss': total_tax_revenue_loss,
        'total_tax_revenue_gain': total_tax_revenue_gain,
        'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_tax_revenue_loss_pct_of_gvt_health_expenditure,
        'average_tax_revenue_loss_pct_of_total_tax_revenues': average_tax_revenue_loss_pct_of_total_tax_revenues
    })

# Convert aggregate results to a DataFrame
results_sample_df = pd.DataFrame(results_sample)

# Save the aggregated results to a CSV or Excel file
results_sample_df.to_csv(f'{output_tables}/Sample_with_original_data_only/SOTJ_sample_aggregate_results.csv', index=False)  # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE

Year 2016: Positive Misalignment: 383927.3348731857, Negative Misalignment: 383927.3348731857, Shifted of total profits: 0.17180618377359663, Total tax revenue loss: 105267.17611630197, Total tax revenue gain: 22683.897196156788
Year 2017: Positive Misalignment: 772911.7255207368, Negative Misalignment: 772911.7255207369, Shifted of total profits: 0.17564830976118664, Total tax revenue loss: 208384.94888698036, Total tax revenue gain: 46422.99498384144
Year 2018: Positive Misalignment: 689664.3892969856, Negative Misalignment: 689664.3892969856, Shifted of total profits: 0.13603676954489405, Total tax revenue loss: 181922.5787613613, Total tax revenue gain: 44738.98137749877
Year 2019: Positive Misalignment: 882689.136865844, Negative Misalignment: 882689.1368658441, Shifted of total profits: 0.1680561686478033, Total tax revenue loss: 228950.4444605462, Total tax revenue gain: 65207.80534852712
Year 2020: Positive Misalignment: 805530.8876180219, Negative Misalignment: 805530.88761802

#### 2.3 Calculate misalignment within EU
- This is for a specific request of a journalist, it can be dropped later

In [13]:
cbcr_eu = cbcr_sample[cbcr_sample['eu'] == 1]

results_eu = []

for year in range(first_year, first_year + n_years):
    print(f"Total profit shifted in USD mn {year} within the EU")
    
    misalignment_year_eu = cbcr_eu[cbcr_eu['year'] == year].copy()
    misalignment_year_eu = calculate_misalignment(misalignment_year_eu, etr_max=0.15, weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0])

    # Keep only the first occurrence of these unique variables for each 'iso_partner'
    unique_columns_eu = misalignment_year_eu.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

    # Perform the groupby operation on 'iso_partner'
    country_results_year_eu = misalignment_year_eu.groupby(['iso_partner']).agg(
        negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
        theoretical_profit=('theoretical_profit', 'sum'),
        reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
    ).reset_index()

    # Convert results to millions
    country_results_year_eu['negative_misalignment'] = -country_results_year_eu['negative_misalignment'] / 1e6
    country_results_year_eu['positive_misalignment'] = country_results_year_eu['positive_misalignment'] / 1e6
    country_results_year_eu['theoretical_profit'] = country_results_year_eu['theoretical_profit'] / 1e6
    country_results_year_eu['reported_profit'] = country_results_year_eu['reported_profit'] / 1e6

    # Merge the unique columns back into the result
    country_results_year_eu = country_results_year_eu.merge(unique_columns_eu, on='iso_partner', how='left')

    # Calculate other relevant variables
    country_results_year_eu['tax_revenue_loss'] = country_results_year_eu['negative_misalignment'] * country_results_year_eu['cit']
    country_results_year_eu['tax_revenue_gain'] = country_results_year_eu['positive_misalignment'] * country_results_year_eu['etr_average_corrected']

    country_results_year_eu['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
        country_results_year_eu['gvt_health_expenditure'] == 0, 
        np.nan, 
        country_results_year_eu['tax_revenue_loss'] / (country_results_year_eu['gvt_health_expenditure'] / 1e6)
    )
    
    country_results_year_eu['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
        country_results_year_eu['tax_revenue_current_usd'] == 0, 
        np.nan, 
        country_results_year_eu['tax_revenue_loss'] / (country_results_year_eu['tax_revenue_current_usd'] / 1e6)
    )

    # Calculate totals
    total_positive_misalignment_eu = country_results_year_eu['positive_misalignment'].sum()
    total_negative_misalignment_eu = country_results_year_eu['negative_misalignment'].sum()
    total_tax_revenue_loss_eu = country_results_year_eu['tax_revenue_loss'].sum()
    total_tax_revenue_gain_eu = country_results_year_eu['tax_revenue_gain'].sum()
    average_tax_revenue_loss_pct_of_gvt_health_expenditure_eu = country_results_year_eu['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
    average_tax_revenue_loss_pct_of_total_tax_revenues_eu = country_results_year_eu['tax_revenue_loss_pct_of_total_tax_revenues'].mean()

    print(f"EU results: Year {year}: Positive Misalignment: {total_positive_misalignment_eu}, Negative Misalignment: {total_negative_misalignment_eu}, "
          f"Total tax revenue loss: {total_tax_revenue_loss_eu}, Total tax revenue gain: {total_tax_revenue_gain_eu}")

    # Calculate countries' fractions of totals
    country_results_year_eu['tax_revenue_loss_caused_pct_of_total'] = country_results_year_eu['positive_misalignment'] / total_positive_misalignment_eu
    country_results_year_eu['tax_revenue_loss_caused_usd'] = country_results_year_eu['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss_eu
    country_results_year_eu['tax_revenue_loss_suffered_pct_of_total'] = country_results_year_eu['tax_revenue_loss'] / total_tax_revenue_loss_eu

    country_results_year_eu = country_results_year_eu[['iso_partner', 'partner_jurisdiction', 'negative_misalignment',
       'tax_revenue_loss', 'tax_revenue_loss_suffered_pct_of_total', 'tax_revenue_loss_pct_of_gvt_health_expenditure',
       'tax_revenue_loss_pct_of_total_tax_revenues', 'positive_misalignment', 'tax_revenue_gain', 
       'tax_revenue_loss_caused_usd', 'tax_revenue_loss_caused_pct_of_total', 'etr_average_corrected', 'cit',
       'region_tjn', 'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

    country_results_year_eu = country_results_year_eu.sort_values(by='iso_partner')
    country_results_year_eu.to_csv(f'{output_tables}/EU_analyses/EU_misalignment_onethirdeach_{year}.csv', index=False)
    
    # Append aggregate results to the list
    results_eu.append({
        'year': year,
        'total_positive_misalignment': total_positive_misalignment_eu,
        'total_negative_misalignment': total_negative_misalignment_eu,
        'total_tax_revenue_loss': total_tax_revenue_loss_eu,
        'total_tax_revenue_gain': total_tax_revenue_gain_eu,
        'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_tax_revenue_loss_pct_of_gvt_health_expenditure_eu,
        'average_tax_revenue_loss_pct_of_total_tax_revenues': average_tax_revenue_loss_pct_of_total_tax_revenues_eu
    })

# Convert aggregate results to a DataFrame
results_eu_df = pd.DataFrame(results_eu)

# Save the aggregated results to a CSV or Excel file
results_eu_df.to_csv(f'{output_tables}/EU_analyses/EU_misalignment_onethirdeach_formula.csv', index=False)

Total profit shifted in USD mn 2016 within the EU
EU results: Year 2016: Positive Misalignment: 150068.17730629534, Negative Misalignment: 150068.1773062953, Total tax revenue loss: 42722.34653517235, Total tax revenue gain: 12772.571802263388
Total profit shifted in USD mn 2017 within the EU


C:\Users\aliso\AppData\Local\Temp\ipykernel_61020\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_61020\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=Fals

EU results: Year 2017: Positive Misalignment: 283629.7645477417, Negative Misalignment: 283629.76454774174, Total tax revenue loss: 78476.35110360313, Total tax revenue gain: 24182.080338703912
Total profit shifted in USD mn 2018 within the EU


C:\Users\aliso\AppData\Local\Temp\ipykernel_61020\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_61020\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=Fals

EU results: Year 2018: Positive Misalignment: 263683.6813885939, Negative Misalignment: 263683.6813885939, Total tax revenue loss: 70759.9611503451, Total tax revenue gain: 22476.201411293536
Total profit shifted in USD mn 2019 within the EU
EU results: Year 2019: Positive Misalignment: 377002.65527275123, Negative Misalignment: 377002.65527275123, Total tax revenue loss: 102150.6792058986, Total tax revenue gain: 34760.47813848504
Total profit shifted in USD mn 2020 within the EU
EU results: Year 2020: Positive Misalignment: 191065.20091755065, Negative Misalignment: 191022.2546562414, Total tax revenue loss: 42453.846923851976, Total tax revenue gain: 18831.275521082265
Total profit shifted in USD mn 2021 within the EU
EU results: Year 2021: Positive Misalignment: 275942.1556109921, Negative Misalignment: 275942.1556109922, Total tax revenue loss: 67554.85497651128, Total tax revenue gain: 24520.714968038712


C:\Users\aliso\AppData\Local\Temp\ipykernel_61020\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_61020\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=Fals

## 3. Calculate misalignment for reporting countries, including the estimated values for their aggregate groups

#### 3.1 Import data

In [20]:
cbcr_sample_filled = pd.read_csv(f'{data_final}/cbcr_with_imputed_values_for_reporting_countries.csv')

FileNotFoundError: [Errno 2] No such file or directory: '../data/2025/final//cbcr_with_imputed_values_for_reporting_countries.csv'

#### 3.2 Calculate misalignment for reporting countries with filled values

In [ ]:
# Initialize a list to store the aggregate results
results_sample_filled = []

for year in range(first_year, first_year + n_years):
    print(f"For correct reporters and their aggregate regions: Total profit shifted in USD mn {year}")
    
    misalignment_year_filled = cbcr_sample_filled[cbcr_sample_filled['year'] == year].copy()
    misalignment_year_filled = calculate_misalignment(misalignment_year_filled, etr_max=0.15, weights=[1/6, 1/3,1/3, 1/6, 0, 0, 0, 0])

    # Keep only the first occurrence of these unique variables for each 'iso_partner'
    unique_columns = misalignment_year_filled.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct']]

    # Perform the groupby operation on 'iso_partner'
    country_results_year_filled = misalignment_year_filled.groupby(['iso_partner']).agg(
        negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
        theoretical_profit=('theoretical_profit', 'sum'),
        reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
    ).reset_index()

    # Convert results to millions
    country_results_year_filled['negative_misalignment'] = -country_results_year_filled['negative_misalignment'] / 1e6
    country_results_year_filled['positive_misalignment'] = country_results_year_filled['positive_misalignment'] / 1e6
    country_results_year_filled['theoretical_profit'] = country_results_year_filled['theoretical_profit'] / 1e6
    country_results_year_filled['reported_profit'] = country_results_year_filled['reported_profit'] / 1e6

    # Merge the unique columns back into the result
    country_results_year_filled = country_results_year_filled.merge(unique_columns, on='iso_partner', how='left')

    # Calculate other relevant variables
    country_results_year_filled['tax_revenue_loss'] = country_results_year_filled['negative_misalignment'] * country_results_year_filled['cit']
    country_results_year_filled['tax_revenue_gain'] = country_results_year_filled['positive_misalignment'] * country_results_year_filled['etr_average_corrected']

    country_results_year_filled['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
        country_results_year_filled['gvt_health_expenditure'] == 0, 
        np.nan, 
        country_results_year_filled['tax_revenue_loss'] / (country_results_year_filled['gvt_health_expenditure'] / 1e6)
    )
    
    country_results_year_filled['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
        country_results_year_filled['tax_revenue_current_usd'] == 0, 
        np.nan, 
        country_results_year_filled['tax_revenue_loss'] / (country_results_year_filled['tax_revenue_current_usd'] / 1e6)
    )

    # Calculate totals
    total_positive_misalignment_filled = country_results_year_filled['positive_misalignment'].sum()
    total_negative_misalignment_filled = country_results_year_filled['negative_misalignment'].sum()
    total_profits_filled = country_results_year_filled['reported_profit'].sum()
    misaligned_of_total_profits_filled = total_positive_misalignment_filled / total_profits_filled
    total_tax_revenue_loss_filled = country_results_year_filled['tax_revenue_loss'].sum()
    total_tax_revenue_gain_filled = country_results_year_filled['tax_revenue_gain'].sum()
    average_tax_revenue_loss_pct_of_gvt_health_expenditure_filled = country_results_year_filled['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
    average_tax_revenue_loss_pct_of_total_tax_revenues_filled = country_results_year_filled['tax_revenue_loss_pct_of_total_tax_revenues'].mean()


    print(f"Year {year}: Positive Misalignment: {total_positive_misalignment_filled}, Negative Misalignment: {total_negative_misalignment_filled}, Shifted of total profits: {misaligned_of_total_profits_filled}, "
          f"Total tax revenue loss: {total_tax_revenue_loss_filled}, Total tax revenue gain: {total_tax_revenue_gain_filled}")

    # Calculate countries' fractions of totals
    country_results_year_filled['tax_revenue_loss_caused_pct_of_total'] = country_results_year_filled['positive_misalignment'] / total_positive_misalignment_filled
    country_results_year_filled['tax_revenue_loss_caused_usd'] = country_results_year_filled['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss_filled
    country_results_year_filled['tax_revenue_loss_suffered_pct_of_total'] = country_results_year_filled['tax_revenue_loss'] / total_tax_revenue_loss_filled

    country_results_year_filled = country_results_year_filled[['iso_partner', 'partner_jurisdiction', 'negative_misalignment',
       'tax_revenue_loss', 'tax_revenue_loss_suffered_pct_of_total', 'tax_revenue_loss_pct_of_gvt_health_expenditure',
       'tax_revenue_loss_pct_of_total_tax_revenues', 'positive_misalignment', 'tax_revenue_gain', 
       'tax_revenue_loss_caused_usd', 'tax_revenue_loss_caused_pct_of_total', 'etr_average_corrected', 'cit',
       'region_tjn', 'ukt', 'oecd', 'oecd_oct']]

    country_results_year_filled = country_results_year_filled.sort_values(by='iso_partner')
    country_results_year_filled.to_csv(f'{output_tables}/Sample_of_original_reporters_filled_missings/CCCTB_{year}_sample_filled.csv', index=False) # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE
    
    # Append aggregate results to the list
    results_sample_filled.append({
        'year': year,
        'total_positive_misalignment': total_positive_misalignment_filled,
        'total_negative_misalignment': total_negative_misalignment_filled,
        'total_profits': total_profits_filled,
        'misaligned_of_total_profits': misaligned_of_total_profits_filled,
        'total_tax_revenue_loss': total_tax_revenue_loss_filled,
        'total_tax_revenue_gain': total_tax_revenue_gain_filled,
        'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_tax_revenue_loss_pct_of_gvt_health_expenditure_filled,
        'average_tax_revenue_loss_pct_of_total_tax_revenues': average_tax_revenue_loss_pct_of_total_tax_revenues_filled
    })

# Convert aggregate results to a DataFrame
results_sample_filled_df = pd.DataFrame(results_sample_filled)

# Save the aggregated results to a CSV or Excel file
results_sample_filled_df.to_csv(f'{output_tables}/CCCTB_sample_filled_aggregate_results.csv', index=False)  # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE

For correct reporters and their aggregate regions: Total profit shifted in USD mn 2016
Year 2016: Positive Misalignment: 1048190.9113828954, Negative Misalignment: 1048190.9113828957, Shifted of total profits: 0.2954265583071442, Total tax revenue loss: 282778.7960361524, Total tax revenue gain: 45058.15802988439
For correct reporters and their aggregate regions: Total profit shifted in USD mn 2017
Year 2017: Positive Misalignment: 1254936.425247722, Negative Misalignment: 1254936.4252477218, Shifted of total profits: 0.17382871346917045, Total tax revenue loss: 328567.365864216, Total tax revenue gain: 71614.20566424313
For correct reporters and their aggregate regions: Total profit shifted in USD mn 2018
Year 2018: Positive Misalignment: 1575904.1745493864, Negative Misalignment: 1575904.1745493864, Shifted of total profits: 0.19089568201135906, Total tax revenue loss: 410237.5038894827, Total tax revenue gain: 79877.16126594903
For correct reporters and their aggregate regions: Tota

/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_misalignment).reset_index(drop=True)
/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_mis

Year 2019: Positive Misalignment: 1752437.7034329618, Negative Misalignment: 1752437.7034329618, Shifted of total profits: 0.2105955401155252, Total tax revenue loss: 443940.58485220897, Total tax revenue gain: 98510.78965054388
For correct reporters and their aggregate regions: Total profit shifted in USD mn 2020
Year 2020: Positive Misalignment: 2007429.3899673605, Negative Misalignment: 2007429.3899673608, Shifted of total profits: 0.3432991415189312, Total tax revenue loss: 492492.5724312198, Total tax revenue gain: 85759.69613183298
For correct reporters and their aggregate regions: Total profit shifted in USD mn 2021
Year 2021: Positive Misalignment: 2284976.137082082, Negative Misalignment: 2284976.1370820818, Shifted of total profits: 0.194073688855372, Total tax revenue loss: 566357.0472328281, Total tax revenue gain: 110713.25261369391


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_misalignment).reset_index(drop=True)
/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_mis

### 4. Calculate misalignment for (one specific) sample with imputed values
- This means that we calculate misalignment for each of the imputed datasets.
- After doing this, we take the median and the 5 and 95 percentiles of the estimated values to get a range of results

#### 4.1 Import data

In [ ]:
cbcr_with_imputed_values = pd.read_csv(f'{data_final}/cbcr_with_imputed_values.csv')

#### 4.2 Calculate misalignment for one specific global sample with imputed values

In [ ]:
# Initialize a list to store the aggregate results
results_global = []

for year in range(first_year, first_year + n_years):
    print(f"Total profit shifted in USD mn {year} based on the imputed sample")
    
    # Filter the dataset for the current year
    misalignment_year_global = cbcr_with_imputed_values[cbcr_with_imputed_values['year'] == year].copy()
    misalignment_year_global = calculate_misalignment(misalignment_year_global, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

    # Keep only the first occurrence of these unique variables for each 'iso_partner'
    unique_columns_global = misalignment_year_global.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct']]

    # Perform the groupby operation on 'iso_partner'
    country_results_year_global = misalignment_year_global.groupby(['iso_partner']).agg(
        negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
        theoretical_profit=('theoretical_profit', 'sum'),
        reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
    ).reset_index()

    # Convert results to millions
    country_results_year_global['negative_misalignment'] = -country_results_year_global['negative_misalignment'] / 1e6
    country_results_year_global['positive_misalignment'] = country_results_year_global['positive_misalignment'] / 1e6
    country_results_year_global['theoretical_profit'] = country_results_year_global['theoretical_profit'] / 1e6
    country_results_year_global['reported_profit'] = country_results_year_global['reported_profit'] / 1e6

    # Merge the unique columns back into the result
    country_results_year_global = country_results_year_global.merge(unique_columns_global, on='iso_partner', how='left')

    # Calculate other relevant variables
    country_results_year_global['tax_revenue_loss'] = country_results_year_global['negative_misalignment'] * country_results_year_global['cit']
    country_results_year_global['tax_revenue_gain'] = country_results_year_global['positive_misalignment'] * country_results_year_global['etr_average_corrected']

    country_results_year_global['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
        country_results_year_global['gvt_health_expenditure'] == 0, 
        np.nan, 
        country_results_year_global['tax_revenue_loss'] / (country_results_year_global['gvt_health_expenditure'] / 1e6)
    )
    
    country_results_year_global['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
        country_results_year_global['tax_revenue_current_usd'] == 0, 
        np.nan, 
        country_results_year_global['tax_revenue_loss'] / (country_results_year_global['tax_revenue_current_usd'] / 1e6)
    )

    # Calculate totals
    total_positive_misalignment_global = country_results_year_global['positive_misalignment'].sum()
    total_negative_misalignment_global = country_results_year_global['negative_misalignment'].sum()
    total_profits_global = country_results_year_global['reported_profit'].sum()
    misaligned_of_total_profits_global = total_positive_misalignment_global / total_profits_global
    total_tax_revenue_loss_global = country_results_year_global['tax_revenue_loss'].sum()
    total_tax_revenue_gain_global = country_results_year_global['tax_revenue_gain'].sum()
    average_tax_revenue_loss_pct_of_gvt_health_expenditure_global = country_results_year_global['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
    average_tax_revenue_loss_pct_of_total_tax_revenues_global = country_results_year_global['tax_revenue_loss_pct_of_total_tax_revenues'].mean()

    print(f"Results with imputed data: Year {year}: Positive Misalignment: {total_positive_misalignment_global}, Negative Misalignment: {total_negative_misalignment_global}, "
          f"Misaliged of total profits: {misaligned_of_total_profits_global},"
          f"Total tax revenue loss: {total_tax_revenue_loss_global}, Total tax revenue gain: {total_tax_revenue_gain_global}")

    # Calculate countries' fractions of totals
    country_results_year_global['tax_revenue_loss_caused_pct_of_total'] = country_results_year_global['positive_misalignment'] / total_positive_misalignment_global
    country_results_year_global['tax_revenue_loss_caused_usd'] = country_results_year_global['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss_global
    country_results_year_global['tax_revenue_loss_suffered_pct_of_total'] = country_results_year_global['tax_revenue_loss'] / total_tax_revenue_loss_global

    country_results_year_global = country_results_year_global[['iso_partner', 'partner_jurisdiction', 'negative_misalignment',
       'tax_revenue_loss', 'tax_revenue_loss_suffered_pct_of_total', 'tax_revenue_loss_pct_of_gvt_health_expenditure',
       'tax_revenue_loss_pct_of_total_tax_revenues', 'positive_misalignment', 'tax_revenue_gain', 
       'tax_revenue_loss_caused_usd', 'tax_revenue_loss_caused_pct_of_total', 'etr_average_corrected', 'cit',
       'region_tjn', 'ukt', 'oecd', 'oecd_oct']]

    country_results_year_global = country_results_year_global.sort_values(by='iso_partner')
    country_results_year_global.to_csv(f'{output_tables}/Global_sample_with_imputed_data/One_sample/sotj_global_sample_{year}.csv', index=False)
    
    # Append aggregate results to the list
    results_global.append({
        'year': year,
        'total_positive_misalignment': total_positive_misalignment_global,
        'total_negative_misalignment': total_negative_misalignment_global,
        'misaligned_of_total_profits': misaligned_of_total_profits_global,
        'total_tax_revenue_loss': total_tax_revenue_loss_global,
        'total_tax_revenue_gain': total_tax_revenue_gain_global,
        'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_tax_revenue_loss_pct_of_gvt_health_expenditure_global,
        'average_tax_revenue_loss_pct_of_total_tax_revenues': average_tax_revenue_loss_pct_of_total_tax_revenues_global
    })

# Convert aggregate results to a DataFrame
results_global_df = pd.DataFrame(results_global)

# Save the aggregated results to a CSV or Excel file
results_global_df.to_csv(f'{output_tables}/Global_sample_with_imputed_data/One_sample/sotj_global_sample_aggregate_results.csv', index=False)

Total profit shifted in USD mn 2016 based on the imputed sample
Results with imputed data: Year 2016: Positive Misalignment: 1193038.060327148, Negative Misalignment: 1193038.060327148, Misaliged of total profits: 0.22624073561511254,Total tax revenue loss: 325661.45965313714, Total tax revenue gain: 55762.44097694043
Total profit shifted in USD mn 2017 based on the imputed sample


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_misalignment).reset_index(drop=True)
/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_mis

Results with imputed data: Year 2017: Positive Misalignment: 1427249.226681822, Negative Misalignment: 1427249.226681822, Misaliged of total profits: 0.17225897628234083,Total tax revenue loss: 386814.52281404915, Total tax revenue gain: 80096.7608433094
Total profit shifted in USD mn 2018 based on the imputed sample
Results with imputed data: Year 2018: Positive Misalignment: 1847459.9911298328, Negative Misalignment: 1847459.9911298333, Misaliged of total profits: 0.21641378530820093,Total tax revenue loss: 483994.224619318, Total tax revenue gain: 95924.5081078072
Total profit shifted in USD mn 2019 based on the imputed sample


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_misalignment).reset_index(drop=True)
/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_mis

Results with imputed data: Year 2019: Positive Misalignment: 1963213.7710531517, Negative Misalignment: 1963213.7710531512, Misaliged of total profits: 0.2342915827925478,Total tax revenue loss: 505895.75928217545, Total tax revenue gain: 107788.36842517092
Total profit shifted in USD mn 2020 based on the imputed sample
Results with imputed data: Year 2020: Positive Misalignment: 2260991.9551695543, Negative Misalignment: 2260991.9551695543, Misaliged of total profits: 0.3846303400808933,Total tax revenue loss: 563349.3285482803, Total tax revenue gain: 101081.41875542907
Total profit shifted in USD mn 2021 based on the imputed sample
Results with imputed data: Year 2021: Positive Misalignment: 2591966.648511404, Negative Misalignment: 2591966.6485114046, Misaliged of total profits: 0.21443742332865123,Total tax revenue loss: 648078.2463460602, Total tax revenue gain: 124574.28312182096


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_misalignment).reset_index(drop=True)
/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_mis

## 5. Calculate misalignment for bootstrapped samples with imputed values and aggregate results

### 5.1 Import data

In [ ]:
cbcr_global_multiple_samples= pd.read_csv(f'{data_final}/cbcr_with_imputed_values_10_samples.csv')

### 5.2 Calculate misalignment for multiple global samples with slightly different imputed values and aggregate results

In [ ]:
# Initialize lists to store results
all_replications = []
aggregate_results = []

# Iterate over each imputation replication and year
for year in range(first_year, first_year + n_years):
    for rep, data in cbcr_global_multiple_samples.groupby("sample_id"):
        print(f"Processing year {year}, replication {rep}")
        
        # Filter the data for the current year and replication
        misalignment_year = data[data['year'] == year].copy()
        misalignment_year = calculate_misalignment(misalignment_year, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

        # Keep only the first occurrence of these unique variables for each 'iso_partner'
        unique_columns = misalignment_year.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                                   'etr_average_corrected', 'cit',
                                                                                   'tax_revenue_current_usd', 
                                                                                   'gvt_health_expenditure', 'region_tjn', 
                                                                                   'ukt', 'oecd', 'oecd_oct']]

        # Perform the groupby operation on 'iso_partner'
        country_results_year = misalignment_year.groupby(['iso_partner']).agg(
            negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
            positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
            theoretical_profit=('theoretical_profit', 'sum'),
            reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
        ).reset_index()

        # Convert results to millions
        country_results_year['negative_misalignment'] = -country_results_year['negative_misalignment'] / 1e6
        country_results_year['positive_misalignment'] = country_results_year['positive_misalignment'] / 1e6
        country_results_year['theoretical_profit'] = country_results_year['theoretical_profit'] / 1e6
        country_results_year['reported_profit'] = country_results_year['reported_profit'] / 1e6

        # Merge the unique columns back into the result
        country_results_year = country_results_year.merge(unique_columns, on='iso_partner', how='left')

        # Calculate other relevant variables
        country_results_year['tax_revenue_loss'] = country_results_year['negative_misalignment'] * country_results_year['cit']
        country_results_year['tax_revenue_gain'] = country_results_year['positive_misalignment'] * country_results_year['etr_average_corrected']

        country_results_year['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
            country_results_year['gvt_health_expenditure'] == 0, 
            np.nan, 
            country_results_year['tax_revenue_loss'] / (country_results_year['gvt_health_expenditure'] / 1e6)
        )

        country_results_year['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
            country_results_year['tax_revenue_current_usd'] == 0, 
            np.nan, 
            country_results_year['tax_revenue_loss'] / (country_results_year['tax_revenue_current_usd'] / 1e6)
        )

        # Add replication number and year
        country_results_year['year'] = year
        country_results_year['n_rep'] = rep

        # Append the replication results to the list
        all_replications.append(country_results_year)

# Concatenate all the results from the replications into a single DataFrame
df_all_replications = pd.concat(all_replications, ignore_index=True)

# Calculate summary statistics (mean, median, std, percentiles)
country_results_bootstrapped = df_all_replications.groupby(['iso_partner', 'year']).agg(
    median_positive_misalignment=('positive_misalignment', 'median'),
    median_negative_misalignment=('negative_misalignment', 'median'),
    avg_positive_misalignment=('positive_misalignment', 'mean'),
    avg_negative_misalignment=('negative_misalignment', 'mean'),
    sd_positive_misalignment=('positive_misalignment', 'std'),
    sd_negative_misalignment=('negative_misalignment', 'std'),
    percentile_2_5_positive_misalignment=('positive_misalignment', lambda x: x.quantile(0.025)),
    percentile_2_5_negative_misalignment=('negative_misalignment', lambda x: x.quantile(0.025)),
    percentile_97_5_positive_misalignment=('positive_misalignment', lambda x: x.quantile(0.975)),
    percentile_97_5_negative_misalignment=('negative_misalignment', lambda x: x.quantile(0.975))
).reset_index()

# Add the actual data from 'cbcr_sample' to replace imputed data
country_results_bootstrapped = country_results_bootstrapped.merge(
    cbcr_sample[['iso_partner', 'year', 'gdp_current_usd', 'gvt_health_expenditure', 'population', 'cit', 'etr_average_corrected']],
    on=['iso_partner', 'year'],
    how='left'
)

# Drop all duplicates
country_results_bootstrapped = country_results_bootstrapped.drop_duplicates(subset=['iso_partner', 'year'])

# Global aggregates for total results across all countries
country_results_bootstrapped['total_avg_positive_misalignment'] = country_results_bootstrapped.groupby('year')['avg_positive_misalignment'].transform('sum')
country_results_bootstrapped['total_avg_negative_misalignment'] = country_results_bootstrapped.groupby('year')['avg_negative_misalignment'].transform('sum')

# Tax revenue loss and gain based on CIT and ETR
country_results_bootstrapped["loss_incurred_cit_usd"] = country_results_bootstrapped["avg_negative_misalignment"] * country_results_bootstrapped["cit"]
country_results_bootstrapped["loss_incurred_etr_usd"] = country_results_bootstrapped["avg_negative_misalignment"] * country_results_bootstrapped["etr_average_corrected"]

# Total losses per year (CIT and ETR)
country_results_bootstrapped["total_loss_cit"] = country_results_bootstrapped.groupby('year')["loss_incurred_cit_usd"].transform('sum')
country_results_bootstrapped["total_loss_etr"] = country_results_bootstrapped.groupby('year')["loss_incurred_etr_usd"].transform('sum')

# Calculate shares of losses and inflicted losses based on misalignment
country_results_bootstrapped["loss_inflicted_share"] = country_results_bootstrapped["avg_positive_misalignment"] / country_results_bootstrapped["total_avg_positive_misalignment"]
country_results_bootstrapped["loss_inflicted_cit_usd"] = country_results_bootstrapped["loss_inflicted_share"] * country_results_bootstrapped["total_loss_cit"]
country_results_bootstrapped["loss_inflicted_etr_usd"] = country_results_bootstrapped["loss_inflicted_share"] * country_results_bootstrapped["total_loss_etr"]

# Save the final dataset
country_results_bootstrapped.to_csv(f'{output_tables}/Global_sample_with_imputed_data/Bootstrapped/SOTJ_country_results_bootstrapped.csv', index=False)

# Aggregate results across all countries for the global output
aggregate_df = country_results_bootstrapped.groupby('year').agg(
    total_positive_misalignment=('total_avg_positive_misalignment', 'mean'),
    total_negative_misalignment=('total_avg_negative_misalignment', 'mean'),
    total_loss_cit=('total_loss_cit', 'mean'),
    total_loss_etr=('total_loss_etr', 'mean')
).reset_index()

# Save global aggregate results
aggregate_df.to_csv(f'{output_tables}/Global_sample_with_imputed_data/Bootstrapped/SOTJ_global_aggregate_results.csv', index=False)

Processing year 2016, replication 1
Processing year 2016, replication 2


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_misalignment).reset_index(drop=True)
/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_mis

Processing year 2016, replication 3
Processing year 2016, replication 4


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_misalignment).reset_index(drop=True)
/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_mis

Processing year 2016, replication 5
Processing year 2016, replication 6


...

/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_misalignment).reset_index(drop=True)
/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_mis

Processing year 2021, replication 7
Processing year 2021, replication 8


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_misalignment).reset_index(drop=True)
/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_mis

Processing year 2021, replication 9
Processing year 2021, replication 10


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_misalignment).reset_index(drop=True)
/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_71059/429223647.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent').apply(adjust_mis

In [ ]:
# Show me results :
country_results_bootstrapped[country_results_bootstrapped['iso_partner'] == 'EGY']


,iso_partner,year,median_positive_misalignment,median_negative_misalignment,avg_positive_misalignment,avg_negative_misalignment,sd_positive_misalignment,sd_negative_misalignment,percentile_2_5_positive_misalignment,percentile_2_5_negative_misalignment,...,etr_average_corrected,total_avg_positive_misalignment,total_avg_negative_misalignment,loss_incurred_cit_usd,loss_incurred_etr_usd,total_loss_cit,total_loss_etr,loss_inflicted_share,loss_inflicted_cit_usd,loss_inflicted_etr_usd
3815,EGY,2016,0.0,2165.541657,0.0,2165.541657,0.0,0.0,0.0,2165.541657,...,0.330637,1.193038e+06,1.193038e+06,487.246873,716.008804,325661.459653,328743.954927,0.0,0.0,0.0
3824,EGY,2017,0.0,2044.599600,0.0,2044.599600,0.0,0.0,0.0,2044.599600,...,0.314165,1.427249e+06,1.427249e+06,460.034910,642.340923,386814.522814,314052.907918,0.0,0.0,0.0
3839,EGY,2018,0.0,1331.884111,0.0,1331.884111,0.0,0.0,0.0,1331.884111,...,0.288416,1.847460e+06,1.847460e+06,299.673925,384.136855,483994.224619,377437.955417,0.0,0.0,0.0
3859,EGY,2019,0.0,2567.332156,0.0,2567.332156,0.0,0.0,0.0,2567.332156,...,0.278343,1.963214e+06,1.963214e+06,577.649735,714.599603,505895.759282,341783.775041,0.0,0.0,0.0
3880,EGY,2020,0.0,3412.670775,0.0,3412.670775,0.0,0.0,0.0,3412.670775,...,0.285301,2.260992e+06,2.260992e+06,767.850924,973.639181,563349.328548,384447.341877,0.0,0.0,0.0
3901,EGY,2021,0.0,8106.011949,0.0,8106.011949,0.0,0.0,0.0,8106.011949,...,0.262068,2.591967e+06,2.591967e+06,1823.852689,2124.324554,648078.246346,446996.331013,0.0,0.0,0.0


In [ ]:

aggregate_df

,year,total_positive_misalignment,total_negative_misalignment,total_loss_cit,total_loss_etr
0,2016,1.193038e+06,1.193038e+06,325661.459653,328743.954927
1,2017,1.427249e+06,1.427249e+06,386814.522814,314052.907918
2,2018,1.847460e+06,1.847460e+06,483994.224619,377437.955417
3,2019,1.963214e+06,1.963214e+06,505895.759282,341783.775041
4,2020,2.260992e+06,2.260992e+06,563349.328548,384447.341877
5,2021,2.591967e+06,2.591967e+06,648078.246346,446996.331013
